## Notebook Workflow Structure

This notebook systematically tests Epic Imaging Reports Annotations extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Setup (imports, random seed) | Environment configured |
| 3 | Cleanup previous outputs | No stale data remains |
| 4 | Start ES container + credentials | Docker running, credentials file generated |
| 5-7 | Populate dummy patient data + ingest Epic Imaging Reports | Documents in Elasticsearch |
| 8 | Index refresh verification | All indices have documents |
| 9-10 | Initialize database and logger | SQLite DB created |
| 11-12 | Create pat2vec config with epic_imaging_reports_annotations mode | Config with correct options |
| 13-14 | Run pat2vec pipeline | Pipeline processes patients successfully |
| 15-16 | Extract all features from database | Features DataFrame populated |
| 17 | Feature preview | Sample of extracted features displayed |
| 18 | Data retrieval test | get_all_features() returns non-empty DataFrame |
| 19-20 | Merge builder functionality | Merge function queries DB and raises ValueError on empty |
| 21 | Cleanup database and project directory | All temp files deleted |
| 22 | Final verification | All assertions pass, TEST SUCCESSFUL |

---
**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- No patient IDs generated after population
- Empty DataFrame from feature extraction


In [ ]:
import os
import random
import shutil
import sys

import numpy as np

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
for dir_to_remove in ["epic_imaging_reports_annotations_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        raise RuntimeError(
            f"Failed to clean up '{dir_to_remove}' directory: {e}. "
            "Critical error - cannot start with stale data."
        ) from e

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container (this may take a few seconds)...")
if not es_container.start():
    raise RuntimeError(
        "Failed to start Elasticsearch container. Check if Docker is running."
    )

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials.py"
creds_content = f"""
username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = "test_files/elastic_schemas.json"

config_populate = config_class(
    proj_name="epic_imaging_reports_annotations_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            raise RuntimeError(f"Index not created: {index}")
    except Exception as e:
        raise RuntimeError(f"Error checking index {index}: {e}")

In [ ]:
import pandas as pd

# Generate and ingest Epic Imaging Reports data to Elasticsearch
from pat2vec.util.get_dummy_data_cohort_searcher import (
    generate_epic_imaging_reports_data,
)
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch

# Generate Epic Imaging Reports data for each patient
imaging_reports_dfs = []
for pid in patient_ids:
    df = generate_epic_imaging_reports_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    imaging_reports_dfs.append(df)

# Combine all Epic Imaging Reports data
df_imaging_reports = (
    pd.concat(imaging_reports_dfs, ignore_index=True)
    if len(imaging_reports_dfs) > 1
    else imaging_reports_dfs[0]
)
df_imaging_reports = df_imaging_reports.where(pd.notnull(df_imaging_reports), None)

# Ingest into Elasticsearch
ingest_data_to_elasticsearch(
    df_imaging_reports,
    "epic_imaging_reports",
    es_client=cs.elastic,
)
cs.elastic.indices.refresh(index="epic_imaging_reports")

print(
    f"Ingested {len(df_imaging_reports)} Epic Imaging Reports documents for {len(patient_ids)} patients"
)

In [ ]:
PROJ_NAME = "epic_imaging_reports_annotations_test_project"
DB_FILENAME = "temp_epic_imaging_reports_annotations_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    raise RuntimeError(
        f"Failed to remove old database file '{DB_PATH}': {e}. "
        "Critical error - cannot start with stale data."
    ) from e

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_imaging_reports_annotations": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

print(
    "pat2vec configuration created with epic_imaging_reports_annotations mode and database backend."
)

In [ ]:
try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: config path invalid. Error details: {e}."
    ) from e
except ValueError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: invalid configuration. Error details: {e}."
    ) from e
except RuntimeError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: initialization failure. Error details: {e}."
    ) from e
except Exception as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: unexpected error. Error details: {e}."
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    raise RuntimeError(
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering."
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    raise RuntimeError(
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    raise RuntimeError(
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    raise RuntimeError(
        "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )

print(f"pat2vec_obj.get_all_features(): {all_features_alt.shape[0]} rows retrieved.")

In [ ]:
from pat2vec.util.post_processing import extract_datetime_to_column

df_with_datetime = extract_datetime_to_column(all_features)

print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {df_with_datetime.shape}")
print(f"Total features: {len(df_with_datetime.columns)}")

if not df_with_datetime.empty:
    print()
    print("First 3 rows:")
    print(df_with_datetime.head(3))
else:
    raise RuntimeError(
        "DataFrame is empty after datetime extraction. Critical error - no features to extract."
    )

In [ ]:
print(
    "\n=== DEMONSTRATING DATA RETRIEVAL FOR EPIC IMAGING REPORTS ANNOTATIONS MODE ==="
)

all_pat_list = pat2vec_obj.all_patient_list

# Get all features (epic_imaging_reports_annotations data is stored in the features table)
from pat2vec.util.helper_functions import get_all_features

data_retrieved = get_all_features(config_obj)

if data_retrieved.empty:
    raise RuntimeError(
        "FATAL ERROR: get_all_features returned empty result. "
        "This indicates a critical failure in epic_imaging_reports_annotations feature extraction."
    )

print(
    f"Retrieved {len(data_retrieved)} row(s) of features for {len(all_pat_list)} patient(s)"
)
print(f"Columns: {list(data_retrieved.columns)[:10]}...")

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("imaging_")]

assert len(feature_cols) > 0, "No feature columns found. Available columns: " + str(
    list(all_features.columns)
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
import pandas as pd


from pat2vec.util.helper_functions import get_all_features

from pat2vec.util.post_processing_build_methods import (
    build_merged_epr_mct_annot_df,
    build_merged_epr_mct_doc_df,
)

print("Merge function imports complete.")

In [ ]:
print("\n=== DEMONSTRATING FEATURE MERGE FUNCTIONALITY ===")


# Call build_merged_epr_mct_annot_df to merge annotation data

merged_path = build_merged_epr_mct_annot_df(
    all_pat_list,
    config_obj,
    overwrite=True,
)


assert merged_path is not None, "build_merged_epr_mct_annot_df should return a path"

assert os.path.exists(
    merged_path
), f"Merged annotations file should exist at {merged_path}"


# Read back and verify non-empty

csv_data = pd.read_csv(merged_path)

assert not csv_data.empty, "CSV file should contain data"


print(f"Merged epic_imaging_reports_annotations data saved to: {merged_path}")

print(f"Shape: {csv_data.shape}")

print(f"Columns: {list(csv_data.columns)[:15]}")


# Test documents merge

docs_path = build_merged_epr_mct_doc_df(
    all_pat_list,
    config_obj,
    overwrite=True,
)


assert docs_path is not None, "build_merged_epr_mct_doc_df should return a path"

assert os.path.exists(docs_path), f"Merged documents file should exist at {docs_path}"


# Read back and verify non-empty

docs_data = pd.read_csv(docs_path)

assert not docs_data.empty, "CSV file should contain data"


print(f"Merged documents saved to: {docs_path}")

print(f"Shape: {docs_data.shape}")

print(f"Columns: {list(docs_data.columns)[:15]}")